# Comparaison U-TILISE vs MAESTRO — Mode aléatoire

Visualisation verticale côte à côte des prédictions U-TILISE et MAESTRO sur le jeu de données aléatoire.

- **U-TILISE** : ne reçoit que les dates claires + dates masquées synthétiquement → pas de prédiction pour les dates nuageuses.
- **MAESTRO** : reçoit toutes les dates (nuageuses incluses) + masquage synthétique → prédiction pour toutes les dates.
- Un **carré gris** signale une date absente pour U-TILISE (date nuageuse non traitée).
- Un **voile violet** distingue les dates masquées synthétiquement.

In [ ]:
import math
import warnings
from pathlib import Path

import matplotlib.patches as mpatches
import matplotlib.pyplot as plt
import numpy as np
import rasterio
import torch
from omegaconf import OmegaConf

from dataloader_CIRCA.datasets.dataset_from_files import Dataset_from_files
from dataloader_CIRCA.tools.data_processor import SentinelDataProcessor
from lib import config_utils, visutils
from lib.eval_tools import Imputation

warnings.filterwarnings("ignore", category=FutureWarning)
%matplotlib inline

## 1. Paramètres

In [ ]:
# ── Zone / fenêtre ─────────────────────────────────────────────────────────────
MGRSC = "31TGJ_row-3_col-2"
WINDOW = (300, 600, 128, 128)  # (col_off, row_off, width, height)

# ── Données ────────────────────────────────────────────────────────────────────
store_dai = Path("/mnt/stores/store_dai")
data_optique = store_dai / "projets/pac/3str/EXP_2/Data_Raster/optique_dataset"
data_radar   = store_dai / "projets/pac/3str/EXP_2/Data_Raster/radar_dataset_v4"
data_masks_aleatoire = store_dai / "projets/pac/3str/EXP_2/Data_Raster/test_v3/aleatoire"

# ── U-TILISE checkpoint ────────────────────────────────────────────────────────
path_ckpt_config = store_dai / "tmp/speillet/cloud_reconstruction_results/U-TILISE/" \
    "v3_mix_closest_random_clouds_combined/2026-03-26_16-17/config.yaml"
path_ckpt_pth = store_dai / "tmp/speillet/cloud_reconstruction_results/U-TILISE/" \
    "v3_mix_closest_random_clouds_combined/2026-03-26_16-17/checkpoints/Model_best.pth"
path_inference_config = Path("./configs/config_run_eval.yaml")

# ── MAESTRO TIF ────────────────────────────────────────────────────────────────
path_maestro_tif = store_dai / "tmp/speillet/inferences_nina/aleatoire/" \
    "bands_stacked_MGRS25-31TGJ_row-3_col-2_ASC_eval.tif"

USE_SAR = "mix_closest"
FILL_VALUE = 1.0
IMAGE_SIZE = [128, 128]
BRIGHTNESS_FACTOR = 3

## 2. Fonctions utilitaires

In [ ]:
def _to_cpu(x):
    if isinstance(x, torch.Tensor):
        return x.detach().cpu().clone()
    elif isinstance(x, dict):
        return {k: _to_cpu(v) for k, v in x.items()}
    elif isinstance(x, (list, tuple)):
        return type(x)(_to_cpu(v) for v in x)
    return x


def sample_to_batch(sample: dict) -> dict:
    batch = {}
    for k, v in sample.items():
        if isinstance(v, torch.Tensor):
            batch[k] = v.unsqueeze(dim=0)
        else:
            batch[k] = v
    return batch


def _to_rgb(tensor_chw, brightness_factor=3):
    """(C, H, W) normalized [0,1] → (H, W, 3) RGB numpy, brightened and clipped."""
    rgb = tensor_chw[[2, 1, 0], :, :].permute(1, 2, 0)  # BGR→RGB
    rgb = (rgb * brightness_factor).clamp(0, 1).numpy()
    return rgb


def _maestro_to_rgb(arr_chw, brightness_factor=3):
    """MAESTRO (4, H, W) uint16 [0‥10000] → (H, W, 3) RGB numpy.
    MAESTRO bands: 0=B, 1=G, 2=R, 3=NIR → RGB = [2, 1, 0]."""
    rgb = arr_chw[[2, 1, 0], :, :].astype(np.float32) / 10000.0
    rgb = np.transpose(rgb, (1, 2, 0))  # (H, W, 3)
    rgb = np.clip(rgb * brightness_factor, 0, 1)
    return rgb


def _gray_square(H, W):
    """Retourne un carré gris (H, W, 3)."""
    return np.full((H, W, 3), 0.5, dtype=np.float32)


def _add_purple_overlay(img, alpha=0.25):
    """Ajoute un voile violet semi-transparent sur une image RGB (H, W, 3)."""
    overlay = img.copy()
    purple = np.array([0.6, 0.2, 0.8], dtype=np.float32)
    overlay = overlay * (1 - alpha) + purple * alpha
    return np.clip(overlay, 0, 1)

## 3. Chargement des données et modèles

In [ ]:
# ── Dataset avec TOUTES les dates (pour alignement MAESTRO) ────────────────────
ds_all = Dataset_from_files(
    mgrsc=MGRSC,
    data_optique=data_optique,
    data_radar=data_radar,
    image_size=IMAGE_SIZE,
    overlap=0,
    fill_value=FILL_VALUE,
    mask_type='fully_masked',
    data_masks=data_masks_aleatoire,
    use_sar=USE_SAR,
    keep_all_dates=True,   # TOUTES les dates → même T que MAESTRO
)
print(f"Dataset (all dates): {len(ds_all)} patches, {ds_all.num_channels} channels")

# ── Dataset filtré (dates claires + synthétiques seulement → U-TILISE) ─────────
ds_rfm = Dataset_from_files(
    mgrsc=MGRSC,
    data_optique=data_optique,
    data_radar=data_radar,
    image_size=IMAGE_SIZE,
    overlap=0,
    fill_value=FILL_VALUE,
    mask_type='fully_masked',
    data_masks=data_masks_aleatoire,
    use_sar=USE_SAR,
    keep_all_dates=False,  # Filtre les nuageuses → sous-ensemble
)
print(f"Dataset (filtered): {len(ds_rfm)} patches, {ds_rfm.num_channels} channels")

In [ ]:
# Modèle U-TILISE
imputation = Imputation(
    config_file_train=str(path_inference_config),
    method="utilise",
    checkpoint=str(path_ckpt_pth),
    config_file_test=str(path_ckpt_config),
    num_channels=ds_rfm.num_channels,
)
print("U-TILISE chargé.")

## 4. Inférence U-TILISE et chargement MAESTRO

In [ ]:
col_off, row_off, width, height = WINDOW

# ── U-TILISE : dataset filtré (sans les dates nuageuses) ──────────────────────
sample_rfm = ds_rfm.get_item_from_mgrsc(
    mgrsc=MGRSC, x=col_off, y=row_off, width=width, height=height,
)
batch_rfm = sample_to_batch(sample_rfm)
batch_rfm, y_pred_rfm = imputation.impute_sample(batch_rfm)

utilise_dates = sample_rfm["S2_dates"]  # dates sur lesquelles U-TILISE a fait une prédiction
utilise_targets = _to_cpu(batch_rfm["y"][0])   # (T_util, 10, H, W)
utilise_inputs  = _to_cpu(batch_rfm["x"][0])   # (T_util, C, H, W)
utilise_preds   = _to_cpu(y_pred_rfm[0])        # (T_util, 10, H, W)

print(f"U-TILISE : {len(utilise_dates)} dates, pred shape {utilise_preds.shape}")

# ── Dataset ALL dates (pour récupérer la liste complète + masques) ─────────────
sample_all = ds_all.get_item_from_mgrsc(
    mgrsc=MGRSC, x=col_off, y=row_off, width=width, height=height,
)
all_dates = sample_all["S2_dates"]          # toutes les T dates
all_targets = _to_cpu(sample_all["y"])       # (T_all, 10, H, W)
all_cloud_mask = _to_cpu(sample_all["cloud_mask"])  # (T_all, 1, H, W)
all_original_masks = _to_cpu(sample_all["original_masks"])  # (T_all, 2, H, W)

T_all = len(all_dates)
print(f"Toutes dates : {T_all}")

# ── Identifier les dates masquées synthétiquement ──────────────────────────────
# cloud_probs > 100 en moyenne → date synthétiquement masquée (+150 offset)
cloud_probs = all_original_masks[:, 0, :, :]  # (T_all, H, W)
is_synthetic = cloud_probs.mean(dim=(1, 2)) > 100  # (T_all,) boolean
synthetic_dates_set = {all_dates[i] for i in range(T_all) if is_synthetic[i]}
print(f"Dates masquées synthétiquement : {len(synthetic_dates_set)}")
print(f"Dates U-TILISE : {len(utilise_dates)} | Dates totales : {T_all}")

In [ ]:
# ── MAESTRO : lecture du TIF ──────────────────────────────────────────────────
with rasterio.open(path_maestro_tif) as src:
    maestro_raw = src.read()  # (T*C, H_full, W_full)

# MAESTRO : 4 bandes (B, G, R, NIR), T_all dates
maestro_all = maestro_raw.reshape((T_all, 4, maestro_raw.shape[1], maestro_raw.shape[2]))

# Extraire la même fenêtre
maestro_window = maestro_all[:, :, row_off:row_off+height, col_off:col_off+width]
print(f"MAESTRO window shape: {maestro_window.shape}  (T, 4, H, W)")

## 5. Visualisation comparative

In [ ]:
def plot_comparison_vertical(
    all_dates,
    all_targets,         # (T_all, 10, H, W) normalized [0,1]
    maestro_window,      # (T_all, 4, H, W) uint16
    utilise_dates,       # list[str] — dates where U-TILISE predicted
    utilise_preds,       # (T_util, 10, H, W) normalized [0,1]
    synthetic_dates_set, # set of date strings that are synthetically masked
    n_cols_groups=2,
    title="",
    brightness_factor=3,
):
    """Visualisation verticale comparative U-TILISE vs MAESTRO.

    Pour chaque date :
      [Target | U-TILISE pred | MAESTRO pred]
    - Carré gris si U-TILISE n'a pas de prédiction pour cette date.
    - Voile violet sur les dates masquées synthétiquement.
    """
    T_all = len(all_dates)
    H, W = all_targets.shape[2], all_targets.shape[3]

    # Build lookup: date → index in U-TILISE predictions
    utilise_date_to_idx = {d: i for i, d in enumerate(utilise_dates)}

    # Pre-build concatenated images per date
    concat_imgs = []
    for t in range(T_all):
        date = all_dates[t]
        is_synth = date in synthetic_dates_set

        # Target RGB
        target_rgb = _to_rgb(all_targets[t], brightness_factor)

        # U-TILISE prediction
        if date in utilise_date_to_idx:
            idx_u = utilise_date_to_idx[date]
            utilise_rgb = _to_rgb(utilise_preds[idx_u], brightness_factor)
        else:
            utilise_rgb = _gray_square(H, W)

        # MAESTRO prediction
        maestro_rgb = _maestro_to_rgb(maestro_window[t], brightness_factor)

        # Apply purple overlay on synthetically masked dates
        if is_synth:
            target_rgb = _add_purple_overlay(target_rgb)
            utilise_rgb = _add_purple_overlay(utilise_rgb)
            maestro_rgb = _add_purple_overlay(maestro_rgb)

        # Concatenate horizontally: Target | U-TILISE | MAESTRO
        row_img = np.concatenate([target_rgb, utilise_rgb, maestro_rgb], axis=1)
        concat_imgs.append(row_img)

    # Layout
    n_rows = math.ceil(T_all / n_cols_groups)
    row_h = 0.55
    col_w = 2.5
    fig_w = col_w * n_cols_groups + 0.15 * n_cols_groups
    fig_h = row_h * n_rows + 0.8

    fig, axes = plt.subplots(
        nrows=n_rows, ncols=n_cols_groups,
        figsize=(fig_w, fig_h),
        gridspec_kw={'wspace': 0.08, 'hspace': 0.0},
    )
    if n_rows == 1 and n_cols_groups == 1:
        axes = np.array([[axes]])
    elif n_rows == 1:
        axes = axes[np.newaxis, :]
    elif n_cols_groups == 1:
        axes = axes[:, np.newaxis]

    fig.suptitle(title, fontsize=8, fontweight='bold', y=0.998)

    for t in range(T_all):
        grp = t // n_rows
        row = t % n_rows
        ax = axes[row, grp]
        ax.imshow(concat_imgs[t])
        ax.set_xticks([])
        ax.set_yticks([])
        for spine in ax.spines.values():
            spine.set_visible(True)
            spine.set_linewidth(0.3)
            spine.set_color('black')

        # Column headers on first row
        if row == 0:
            for i, lbl in enumerate(['Target', 'U-TILISE', 'MAESTRO']):
                x_frac = (i + 0.5) / 3
                ax.text(x_frac, 1.02, lbl, fontsize=5, fontweight='bold',
                        ha='center', va='bottom', transform=ax.transAxes)

        # Date label
        ax.annotate(all_dates[t], xy=(0, 0.5), xycoords='axes fraction',
                    xytext=(-2, 0), textcoords='offset points',
                    fontsize=4.5, ha='right', va='center', rotation=-35)

    # Hide unused axes
    remainder = T_all % n_rows
    if remainder != 0:
        for row in range(remainder, n_rows):
            axes[row, n_cols_groups - 1].set_visible(False)

    # Legend
    legend_elements = [
        mpatches.Patch(facecolor=(0.5, 0.5, 0.5), edgecolor='black',
                       linewidth=0.5, label='Pas de prédiction U-TILISE'),
        mpatches.Patch(facecolor=(0.75, 0.6, 0.85), edgecolor='black',
                       linewidth=0.5, label='Date masquée synthétiquement'),
    ]
    fig.legend(handles=legend_elements, loc='lower center', ncol=2,
              fontsize=6, frameon=True, fancybox=True,
              bbox_to_anchor=(0.5, -0.005))

    fig.subplots_adjust(left=0.06, right=0.995, top=0.96, bottom=0.02)
    plt.show()

In [ ]:
plot_comparison_vertical(
    all_dates=all_dates,
    all_targets=all_targets,
    maestro_window=maestro_window,
    utilise_dates=utilise_dates,
    utilise_preds=utilise_preds,
    synthetic_dates_set=synthetic_dates_set,
    n_cols_groups=4,
    title=f"Comparaison U-TILISE vs MAESTRO — Mode aléatoire — {MGRSC} — fenêtre {WINDOW}",
    brightness_factor=BRIGHTNESS_FACTOR,
)

## 6. Résumé des dates

Tableau récapitulatif des dates avec leur statut.

In [ ]:
utilise_set = set(utilise_dates)

print(f"{'Date':>12}  {'Synthétique':>12}  {'U-TILISE':>10}  {'MAESTRO':>10}")
print("-" * 52)
for t, date in enumerate(all_dates):
    is_synth = "OUI" if date in synthetic_dates_set else ""
    has_util = "✓" if date in utilise_set else "✗ (absent)"
    has_mae  = "✓"
    print(f"{date:>12}  {is_synth:>12}  {has_util:>10}  {has_mae:>10}")

print(f"\nTotal : {T_all} dates | U-TILISE : {len(utilise_dates)} | MAESTRO : {T_all}")
print(f"Dates masquées synthétiquement : {len(synthetic_dates_set)}")
print(f"Dates absentes U-TILISE : {T_all - len(utilise_dates)}")